In [ ]:
!pip install antspyx, trimesh, fast_simplification

In [ ]:
import os
import sys
import ants
import trimesh
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
img = nib.load("path/to/ixi-t1-mri-file")
data = img.get_fdata()

print("Shape:", img.shape)
print("Voxel spacing:", img.header.get_zooms())
print("Data type:", img.get_data_dtype())
print("Affine:")
print(img.affine)

print("Minimum intensity:", np.min(data))
print("Maximum intensity:", np.max(data))
print("Mean intensity:", np.mean(data))
print("Non-zero voxels:", np.count_nonzero(data))

In [ ]:

mid_x = data.shape[0] // 2
mid_y = data.shape[1] // 2
mid_z = data.shape[2] // 2

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(data[mid_x, :, :].T, cmap="gray", origin="lower")
plt.title("Sagittal")

plt.subplot(1, 3, 2)
plt.imshow(data[:, mid_y, :].T, cmap="gray", origin="lower")
plt.title("Coronal")

plt.subplot(1, 3, 3)
plt.imshow(data[:, :, mid_z].T, cmap="gray", origin="lower")
plt.title("Axial")

plt.show()

In [ ]:
img = nib.load("path/to/ixi-t1-mri-file")

print("Shape:", img.shape)
print("Voxel spacing:", img.header.get_zooms())

print("\nOrientation:")
print(nib.aff2axcodes(img.affine))

print("\nCanonical orientation:")
canonical = nib.as_closest_canonical(img)
print("Shape:", canonical.shape)
print("Orientation:", nib.aff2axcodes(canonical.affine))

print("\nqform:")
print(img.get_qform())

print("\nsform:")
print(img.get_sform())

print("\nqform_code:", img.header["qform_code"])
print("sform_code:", img.header["sform_code"])

In [ ]:
print("\nHEADER")
print("Shape:", img.shape)
print("Spacing:", img.header.get_zooms()[:3])
print("Orientation:", nib.aff2axcodes(img.affine))

print("\nAFFINE")
print(img.affine)

print("\nWORLD BOUNDS")

# All 8 corners
corners = np.array([
    [0, 0, 0, 1],
    [img.shape[0]-1, 0, 0, 1],
    [0, img.shape[1]-1, 0, 1],
    [0, 0, img.shape[2]-1, 1],
    [img.shape[0]-1, img.shape[1]-1, 0, 1],
    [img.shape[0]-1, 0, img.shape[2]-1, 1],
    [0, img.shape[1]-1, img.shape[2]-1, 1],
    [img.shape[0]-1, img.shape[1]-1, img.shape[2]-1, 1]
])

world = (img.affine @ corners.T).T[:, :3]

print("Min:", world.min(axis=0))
print("Max:", world.max(axis=0))

In [ ]:
template_path = "path/to/mni305-template/average305_t1_tal_lin.nii"

template_img = nib.load(template_path)
template = template_img.get_fdata()

print(template.shape)

x = template.shape[0] // 2
y = template.shape[1] // 2
z = template.shape[2] // 2

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(template[x, :, :].T, cmap="gray", origin="lower")
plt.title("Sagittal")

plt.subplot(1, 3, 2)
plt.imshow(template[:, y, :].T, cmap="gray", origin="lower")
plt.title("Coronal")

plt.subplot(1, 3, 3)
plt.imshow(template[:, :, z].T, cmap="gray", origin="lower")
plt.title("Axial")

plt.show()

In [ ]:
def reorient_to_ras(input_path, output_path):
    # Load image
    img = nib.load(input_path)

    # Determine current orientation
    current_orientation = nib.aff2axcodes(img.affine)

    print("Input:", os.path.basename(input_path))
    print("Original orientation:", current_orientation)
    print("Original shape:", img.shape)

    # Calculate transformation from current orientation -> RAS
    ras_orientation = nib.orientations.axcodes2ornt(("R", "A", "S"))
    current_ornt = nib.orientations.io_orientation(img.affine)

    transform = nib.orientations.ornt_transform(
        current_ornt,
        ras_orientation
    )

    # Reorient voxel data
    data_ras = nib.orientations.apply_orientation(
        img.get_fdata(),
        transform
    )

    # Calculate new affine
    affine_ras = img.affine @ nib.orientations.inv_ornt_aff(
        transform,
        img.shape
    )

    # Create new NIfTI
    ras_img = nib.Nifti1Image(
        data_ras,
        affine_ras,
        img.header.copy()
    )

    # Make sure header reflects the new data shape
    ras_img.header.set_data_shape(data_ras.shape)

    # Save
    nib.save(ras_img, output_path)

    # Verify
    check = nib.load(output_path)

    print("RAS orientation:", nib.aff2axcodes(check.affine))
    print("RAS shape:", check.shape)
    print("Saved:", output_path)
    print("-" * 60)

    return output_path

In [ ]:
raw_t1 = "path/to/ixi-t1-mri-file"

output_dir = "path/to/output-folder"
os.makedirs(output_dir, exist_ok=True)

ras_t1 = os.path.join(
    output_dir,
    "output_T1_RAS.nii.gz"
)

reorient_to_ras(raw_t1, ras_t1)

In [ ]:
moving_path = "path/to/output_T1_RAS.nii.gz"

fixed_path = "path/to/mni305-template/average305_t1_tal_lin.nii"

In [ ]:
fixed = ants.image_read(fixed_path)
moving = ants.image_read(moving_path)

In [ ]:

registration = ants.registration(
    fixed=fixed,
    moving=moving,
    type_of_transform="Affine"
)

In [ ]:
registered = registration["warpedmovout"]

In [ ]:
output_dir = "path/to/MNI305"
os.makedirs(output_dir, exist_ok=True)

registered_path = os.path.join(
    output_dir,
    "output_T1_MNI305.nii.gz"
)

ants.image_write(
    registered,
    registered_path
)

print("Saved:", registered_path)

In [ ]:
import shutil

transform_src = registration["fwdtransforms"][0]

transform_dir = "path/to/MNI305/transforms"
os.makedirs(transform_dir, exist_ok=True)

transform_path = os.path.join(
    transform_dir,
    "output_to_MNI305_Affine.mat"
)

shutil.copy2(transform_src, transform_path)

print("Saved transform:", transform_path)

In [ ]:
fixed_np = fixed.numpy()
moving_np = registered.numpy()

# Middle slices
x = fixed_np.shape[0] // 2
y = fixed_np.shape[1] // 2
z = fixed_np.shape[2] // 2

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Sagittal
axes[0].imshow(
    fixed_np[x, :, :].T,
    cmap="gray",
    origin="lower"
)
axes[0].imshow(
    moving_np[x, :, :].T,
    cmap="Reds",
    alpha=0.35,
    origin="lower"
)
axes[0].set_title("Sagittal overlay")

# Coronal
axes[1].imshow(
    fixed_np[:, y, :].T,
    cmap="gray",
    origin="lower"
)
axes[1].imshow(
    moving_np[:, y, :].T,
    cmap="Reds",
    alpha=0.35,
    origin="lower"
)
axes[1].set_title("Coronal overlay")

# Axial
axes[2].imshow(
    fixed_np[:, :, z].T,
    cmap="gray",
    origin="lower"
)
axes[2].imshow(
    moving_np[:, :, z].T,
    cmap="Reds",
    alpha=0.35,
    origin="lower"
)
axes[2].set_title("Axial overlay")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
!pip install -q "monai[all]"

In [ ]:
from monai.bundle import download

download(
    name="wholeBrainSeg_Large_UNEST_segmentation",
    bundle_dir="path/to/wholeBrainSeg_Large_UNEST_segmentation"
)

In [ ]:
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    NormalizeIntensityd,
    EnsureTyped
)

preprocess = Compose([
    LoadImaged(
        keys=["image"]
    ),
    EnsureChannelFirstd(
        keys=["image"]
    ),
    NormalizeIntensityd(
        keys=["image"],
        nonzero=True,
        channel_wise=True
    ),
    EnsureTyped(
        keys=["image"]
    )
])

In [ ]:
registered_path = "path/to/MNI305/IXI351_T1_MNI305.nii.gz"

data = {
    "image": registered_path
}

data = preprocess(data)

print("Keys:", data.keys())
print("Image shape:", data["image"].shape)
print("Image dtype:", data["image"].dtype)

In [ ]:
import sys
import torch

bundle_root = "path/to/wholeBrainSeg_Large_UNEST_segmentation/wholeBrainSeg_Large_UNEST_segmentation"

if bundle_root not in sys.path:
    sys.path.insert(0, bundle_root)

from scripts.networks.unest_base_patch_4 import UNesT

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

In [ ]:
network = UNesT(
    in_channels=1,
    out_channels=133,
    patch_size=4,
    depths=[2, 2, 8],
    embed_dim=[128, 256, 512],
    num_heads=[4, 8, 16]
).to(device)

print("UNesT model created.")

In [ ]:
model_path = f"{bundle_root}/models/model.pt"

checkpoint = torch.load(
    model_path,
    map_location=device
)


network.load_state_dict(
    checkpoint["model"],
    strict=True
)

network.eval()

print("Model is ready for inference.")

In [ ]:
from monai.inferers import SlidingWindowInferer

inferer = SlidingWindowInferer(
    roi_size=(96, 96, 96),
    sw_batch_size=1,
    overlap=0.7
)

print("Inferer created.")

In [ ]:
input_tensor = data["image"].unsqueeze(0).to(device)

In [ ]:
network.eval()

with torch.no_grad():
    prediction = inferer(
        inputs=input_tensor,
        network=network
    )

print("Prediction shape:", prediction.shape)

In [ ]:
prediction_labels = torch.argmax(
    prediction,
    dim=1
)

print("Label map shape:", prediction_labels.shape)
print("Label dtype:", prediction_labels.dtype)

In [ ]:
import nibabel as nib
import numpy as np

registered_path = "path/to/MNI305/IXI351_T1_MNI305.nii.gz"

reference_img = nib.load(registered_path)

labels_np = prediction_labels[0].cpu().numpy().astype(np.uint8)

segmentation_path = (
    "path/to/MNI305/"
    "IXI351_UNEST_segmentation_MNI305.nii.gz"
)

seg_img = nib.Nifti1Image(
    labels_np,
    reference_img.affine,
    reference_img.header.copy()
)

seg_img.set_data_dtype(np.uint8)

nib.save(seg_img, segmentation_path)

print("Saved:", segmentation_path)

In [ ]:
import json
import pandas as pd

metadata_path = f"{bundle_root}/configs/metadata.json"

with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

label_dict = metadata["network_data_format"]["outputs"]["pred"]["channel_def"]

rows = []

for label_id, label_name in label_dict.items():

    # Extract laterality
    if label_name.startswith("Right-"):
        laterality = "RIGHT"
        anatomical_name = label_name[len("Right-"):]

    elif label_name.startswith("Left-"):
        laterality = "LEFT"
        anatomical_name = label_name[len("Left-"):]

    else:
        laterality = "NONE"
        anatomical_name = label_name

    rows.append({
        "source": "MONAI_UNEST",
        "source_version": metadata["version"],
        "source_id": int(label_id),
        "source_name": label_name,
        "anatomical_name": anatomical_name,
        "laterality": laterality
    })

unest_labels = pd.DataFrame(rows)

print(unest_labels.to_string(index=False))
print("\nNumber of labels:", len(unest_labels))

In [ ]:
unest_labels_path = "path/to/MNI305/unest_source_labels.csv"

unest_labels.to_csv(
    unest_labels_path,
    index=False
)

print("Saved:", unest_labels_path)

In [ ]:
import re
import pandas as pd


def make_canonical_id(name):
    """
    Convert an anatomical name into a stable canonical-style ID.
    Example:
        Hippocampus -> BRN-HIP
        Angular Gyrus -> BRN-ANGULAR-GYR
    """

    name = name.strip().lower()

    # Remove source abbreviations such as ACgG--, MFG--- etc.
    if "--" in name:
        name = name.split("--", 1)[-1]
    elif "---" in name:
        name = name.split("---", 1)[-1]

    # Clean punctuation
    name = re.sub(r"[^a-z0-9]+", "-", name)
    name = name.strip("-")

    # Some common canonical abbreviations
    replacements = {
        "hippocampus": "HIP",
        "amygdala": "AMY",
        "thalamus-proper": "THA",
        "caudate": "CAU",
        "putamen": "PUT",
        "pallidum": "PAL",
        "precuneus": "PCU",
        "precentral-gyrus": "PRG",
        "postcentral-gyrus": "POG",
        "brain-stem": "BST",
        "third-ventricle": "3V",
        "fourth-ventricle": "4V",
        "accumbens-area": "ACC",
        "cerebellum-exterior": "CB-EXT",
        "cerebellum-white-matter": "CB-WM",
        "cerebral-white-matter": "CER-WM",
        "lateral-ventricle": "LAT-V",
        "inf-lat-vent": "INF-LAT-V",
        "ventral-dc": "VDC",
        "basal-forebrain": "BASAL-FOREBRAIN",
        "background": "BACKGROUND",
    }

    if name in replacements:
        return "BRN-" + replacements[name]

    return "BRN-" + name.upper()




concept_rows = []

for _, row in unest_labels.iterrows():

    anatomical_name = row["anatomical_name"].strip()

    if anatomical_name.lower() == "background":
        continue

    canonical_id = make_canonical_id(anatomical_name)

    concept_rows.append({
        "canonical_id": canonical_id,
        "preferred_name": anatomical_name,
        "parent_id": "BRN",
        "category": "anatomical_structure",
        "laterality_allowed": row["laterality"] != "NONE",
        "description": f"Canonical concept derived from UNesT source term: {anatomical_name}"
    })


# Remove duplicates
canonical_concepts_full = (
    pd.DataFrame(concept_rows)
    .drop_duplicates(subset=["canonical_id"])
    .reset_index(drop=True)
)


canonical_path_full = (
    "path/to/MNI305/canonical_concepts_full.csv"
)

canonical_concepts_full.to_csv(
    canonical_path_full,
    index=False
)


print("Canonical concepts created:", len(canonical_concepts_full))
print("Saved:", canonical_path_full)


In [ ]:

source_mappings_full = []

for _, row in unest_labels.iterrows():

    source_name = row["source_name"]
    anatomical_name = row["anatomical_name"]
    laterality = row["laterality"]

    if anatomical_name.lower() == "background":

        source_mappings_full.append({
            "source": row["source"],
            "source_version": row["source_version"],
            "source_id": row["source_id"],
            "source_name": source_name,
            "canonical_id": "BRN-BACKGROUND",
            "laterality": "NONE",
            "mapping_type": "system",
            "mapping_confidence": 1.0,
            "review_status": "APPROVED",
            "notes": "Background / non-anatomical voxel"
        })

        continue


    canonical_id = make_canonical_id(anatomical_name)

    mapping_type = "exact"
    confidence = 1.0
    review_status = "APPROVED"
    notes = ""


    # ---------------------------------------------
    # Known potentially duplicated subdivisions
    # ---------------------------------------------

    if anatomical_name.lower() in [
        "mpog--postcentral-gyrus",
        "pog---postcentral-gyrus",
        "mprg--precentral-gyrus",
        "prg---precentral-gyrus"
    ]:

        mapping_type = "review_required"
        confidence = None
        review_status = "REVIEW"

        notes = (
            "Source contains potentially distinct cortical "
            "subdivisions. Preserve separately until the "
            "underlying parcellation is verified."
        )


    source_mappings_full.append({

        "source": row["source"],
        "source_version": row["source_version"],
        "source_id": row["source_id"],
        "source_name": source_name,

        "canonical_id": canonical_id,

        "laterality": laterality,

        "mapping_type": mapping_type,

        "mapping_confidence": confidence,

        "review_status": review_status,

        "notes": notes
    })


source_mappings_full = pd.DataFrame(source_mappings_full)


mapping_path_full = (
    "path/to/MNI305/source_mappings_full.csv"
)

source_mappings_full.to_csv(
    mapping_path_full,
    index=False
)


print("Total source labels:", len(source_mappings_full))


In [ ]:
# Add the two anatomically distinct medial-segment concepts

new_concepts = pd.DataFrame([
    {
        "canonical_id": "BRN-POG-MED",
        "preferred_name": "Postcentral Gyrus, Medial Segment",
        "parent_id": "BRN-POG",
        "category": "cortical_structure",
        "laterality_allowed": True,
        "description": "Medial segment of the postcentral gyrus."
    },
    {
        "canonical_id": "BRN-PRG-MED",
        "preferred_name": "Precentral Gyrus, Medial Segment",
        "parent_id": "BRN-PRG",
        "category": "cortical_structure",
        "laterality_allowed": True,
        "description": "Medial segment of the precentral gyrus."
    }
])

canonical_concepts_full = pd.concat(
    [canonical_concepts_full, new_concepts],
    ignore_index=True
)
# Remove accidental duplicates if this cell is run more than once
canonical_concepts_full = canonical_concepts_full.drop_duplicates(
    subset=["canonical_id"],
    keep="first"
).reset_index(drop=True)


In [ ]:
review_mapping_updates = {
    77: {
        "canonical_id": "BRN-POG-MED",
        "laterality": "RIGHT",
        "mapping_type": "exact",
        "mapping_confidence": 1.0,
        "review_status": "APPROVED",
        "notes": "MPoG = postcentral gyrus, medial segment."
    },
    
    78: {
        "canonical_id": "BRN-POG-MED",
        "laterality": "LEFT",
        "mapping_type": "exact",
        "mapping_confidence": 1.0,
        "review_status": "APPROVED",
        "notes": "MPoG = postcentral gyrus, medial segment."
    },
    
    79: {
        "canonical_id": "BRN-PRG-MED",
        "laterality": "RIGHT",
        "mapping_type": "exact",
        "mapping_confidence": 1.0,
        "review_status": "APPROVED",
        "notes": "MPrG = precentral gyrus, medial segment."
    },
    
    80: {
        "canonical_id": "BRN-PRG-MED",
        "laterality": "LEFT",
        "mapping_type": "exact",
        "mapping_confidence": 1.0,
        "review_status": "APPROVED",
        "notes": "MPrG = precentral gyrus, medial segment."
    },
    
    103: {
        "canonical_id": "BRN-POG",
        "laterality": "RIGHT",
        "mapping_type": "exact",
        "mapping_confidence": 1.0,
        "review_status": "APPROVED",
        "notes": "PoG = postcentral gyrus."
    },
    
    104: {
        "canonical_id": "BRN-POG",
        "laterality": "LEFT",
        "mapping_type": "exact",
        "mapping_confidence": 1.0,
        "review_status": "APPROVED",
        "notes": "PoG = postcentral gyrus."
    },
    
    109: {
        "canonical_id": "BRN-PRG",
        "laterality": "RIGHT",
        "mapping_type": "exact",
        "mapping_confidence": 1.0,
        "review_status": "APPROVED",
        "notes": "PrG = precentral gyrus."
    },
    
    110: {
        "canonical_id": "BRN-PRG",
        "laterality": "LEFT",
        "mapping_type": "exact",
        "mapping_confidence": 1.0,
        "review_status": "APPROVED",
        "notes": "PrG = precentral gyrus."
    }
}

In [ ]:
for source_id, update in review_mapping_updates.items():
    mask = source_mappings_full["source_id"] == source_id
    
    for column, value in update.items():
        source_mappings_full.loc[mask, column] = value


In [ ]:
canonical_concepts_v2 = canonical_concepts_full.copy()

In [ ]:
hierarchy_updates = {
    
    # Brain
    "BRN": {
        "preferred_name": "Brain",
        "parent_id": None,
        "category": "organ"
    },

    # Ventricular system
    "BRN-3V": {
        "parent_id": "BRN",
        "category": "ventricle"
    },

    "BRN-4V": {
        "parent_id": "BRN",
        "category": "ventricle"
    },

    # Brain stem
    "BRN-BST": {
        "parent_id": "BRN",
        "category": "brainstem"
    },

    # Deep gray matter
    "BRN-HIP": {
        "parent_id": "BRN",
        "category": "deep_gray_matter"
    },

    "BRN-AMY": {
        "parent_id": "BRN",
        "category": "deep_gray_matter"
    },

    "BRN-THA": {
        "parent_id": "BRN",
        "category": "deep_gray_matter"
    },

    "BRN-CAU": {
        "parent_id": "BRN",
        "category": "deep_gray_matter"
    },

    "BRN-PUT": {
        "parent_id": "BRN",
        "category": "deep_gray_matter"
    },

    "BRN-PAL": {
        "parent_id": "BRN",
        "category": "deep_gray_matter"
    },

    # Motor/sensory cortical structures
    "BRN-PCU": {
        "parent_id": "BRN",
        "category": "cortical_structure"
    },

    "BRN-PRG": {
        "parent_id": "BRN",
        "category": "cortical_structure"
    },

    "BRN-POG": {
        "parent_id": "BRN",
        "category": "cortical_structure"
    },

    "BRN-PRG-MED": {
        "parent_id": "BRN-PRG",
        "category": "cortical_subregion"
    },

    "BRN-POG-MED": {
        "parent_id": "BRN-POG",
        "category": "cortical_subregion"
    }
}

In [ ]:
for canonical_id, update in hierarchy_updates.items():

    mask = canonical_concepts_v2["canonical_id"] == canonical_id

    for column, value in update.items():
        canonical_concepts_v2.loc[mask, column] = value

In [ ]:
import pandas as pd

parent_concepts = pd.DataFrame([
    ["BRN-CER", "Cerebrum", "BRN", "major_region"],
    ["BRN-CER-COR", "Cerebral Cortex", "BRN-CER", "cortex"],
    ["BRN-FRO", "Frontal Lobe", "BRN-CER-COR", "cortical_lobe"],
    ["BRN-PAR", "Parietal Lobe", "BRN-CER-COR", "cortical_lobe"],
    ["BRN-TEM", "Temporal Lobe", "BRN-CER-COR", "cortical_lobe"],
    ["BRN-OCC", "Occipital Lobe", "BRN-CER-COR", "cortical_lobe"],
    ["BRN-LIM", "Limbic Region", "BRN-CER-COR", "cortical_region"],
    ["BRN-INS", "Insular Cortex", "BRN-CER-COR", "cortical_region"],
    ["BRN-DGM", "Deep Gray Matter", "BRN-CER", "deep_gray_matter"],
    ["BRN-WM", "Cerebral White Matter", "BRN-CER", "white_matter"],
    ["BRN-CBL", "Cerebellum", "BRN", "major_region"],
    ["BRN-VEN", "Ventricular System", "BRN", "major_region"],
    ["BRN-BST-GRP", "Brainstem", "BRN", "major_region"],
], columns=[
    "canonical_id",
    "preferred_name",
    "parent_id",
    "category"
])

parent_concepts["ontology_level"] = (
    parent_concepts["parent_id"]
    .map({"BRN": 1, "BRN-CER": 2, "BRN-CER-COR": 3})
    .fillna(1)
)

parent_concepts["laterality_allowed"] = False
parent_concepts["description"] = ""

canonical_concepts_v2 = pd.concat(
    [canonical_concepts_v2, parent_concepts],
    ignore_index=True
).drop_duplicates("canonical_id")

In [ ]:
# ============================================================
# CANONICAL ONTOLOGY V2 — BUILD HIERARCHY
# ============================================================

import pandas as pd

# Start from your current ontology
ontology = canonical_concepts_v2.copy()

# ------------------------------------------------------------
# 1. Helper: assign parent to a concept
# ------------------------------------------------------------
def set_parent(ontology, canonical_ids, parent_id):
    ontology.loc[
        ontology["canonical_id"].isin(canonical_ids),
        "parent_id"
    ] = parent_id
    return ontology

# ------------------------------------------------------------
# 2. High-level parents
# ------------------------------------------------------------

# Cerebral cortex
cortex_ids = [
    "BRN-ANGULAR-GYRUS",
    "BRN-ANTERIOR-CINGULATE-GYRUS",
    "BRN-ANTERIOR-INSULA",
    "BRN-ANTERIOR-ORBITAL-GYRUS",
    "BRN-CALCARINE-CORTEX",
    "BRN-CENTRAL-OPERCULUM",
    "BRN-FRONTAL-OPERCULUM",
    "BRN-FRONTAL-POLE",
    "BRN-FUSIFORM-GYRUS",
    "BRN-GYRUS-RECTUS",
    "BRN-INFERIOR-OCCIPITAL-GYRUS",
    "BRN-INFERIOR-TEMPORAL-GYRUS",
    "BRN-LATERAL-ORBITAL-GYRUS",
    "BRN-LINGUAL-GYRUS",
    "BRN-MEDIAL-FRONTAL-CORTEX",
    "BRN-MEDIAL-ORBITAL-GYRUS",
    "BRN-MIDDLE-CINGULATE-GYRUS",
    "BRN-MIDDLE-FRONTAL-GYRUS",
    "BRN-MIDDLE-OCCIPITAL-GYRUS",
    "BRN-MIDDLE-TEMPORAL-GYRUS",
    "BRN-OCCIPITAL-FUSIFORM-GYRUS",
    "BRN-OCCIPITAL-POLE",
    "BRN-PARAHIPPOCAMPAL-GYRUS",
    "BRN-PARIETAL-OPERCULUM",
    "BRN-PCU",
    "BRN-PLANUM-POLARE",
    "BRN-PLANUM-TEMPORALE",
    "BRN-POG",
    "BRN-POG-MED",
    "BRN-POSTERIOR-CINGULATE-GYRUS",
    "BRN-POSTERIOR-INSULA",
    "BRN-POSTERIOR-ORBITAL-GYRUS",
    "BRN-PRG",
    "BRN-PRG-MED",
    "BRN-SUBCALLOSAL-AREA",
    "BRN-SUPERIOR-FRONTAL-GYRUS",
    "BRN-SUPERIOR-OCCIPITAL-GYRUS",
    "BRN-SUPERIOR-TEMPORAL-GYRUS",
    "BRN-SUPPLEMENTARY-MOTOR-CORTEX",
    "BRN-SUPRAMARGINAL-GYRUS",
    "BRN-TEMPORAL-POLE",
    "BRN-TRANSVERSE-TEMPORAL-GYRUS",
]

ontology = set_parent(
    ontology,
    cortex_ids,
    "BRN-CER-COR"
)

# ------------------------------------------------------------
# 3. Frontal lobe
# ------------------------------------------------------------

frontal_ids = [
    "BRN-ANTERIOR-ORBITAL-GYRUS",
    "BRN-FRONTAL-OPERCULUM",
    "BRN-FRONTAL-POLE",
    "BRN-GYRUS-RECTUS",
    "BRN-LATERAL-ORBITAL-GYRUS",
    "BRN-MEDIAL-FRONTAL-CORTEX",
    "BRN-MEDIAL-ORBITAL-GYRUS",
    "BRN-MIDDLE-FRONTAL-GYRUS",
    "BRN-POSTERIOR-ORBITAL-GYRUS",
    "BRN-PRG",
    "BRN-PRG-MED",
    "BRN-SUPERIOR-FRONTAL-GYRUS",
    "BRN-SUPPLEMENTARY-MOTOR-CORTEX",
]

ontology = set_parent(
    ontology,
    frontal_ids,
    "BRN-FRO"
)

# ------------------------------------------------------------
# 4. Parietal lobe
# ------------------------------------------------------------

parietal_ids = [
    "BRN-ANGULAR-GYRUS",
    "BRN-PARIETAL-OPERCULUM",
    "BRN-PCU",
    "BRN-POG",
    "BRN-POG-MED",
    "BRN-SUPRAMARGINAL-GYRUS",
]

ontology = set_parent(
    ontology,
    parietal_ids,
    "BRN-PAR"
)

# ------------------------------------------------------------
# 5. Temporal lobe
# ------------------------------------------------------------

temporal_ids = [
    "BRN-FUSIFORM-GYRUS",
    "BRN-INFERIOR-TEMPORAL-GYRUS",
    "BRN-MIDDLE-TEMPORAL-GYRUS",
    "BRN-PLANUM-POLARE",
    "BRN-PLANUM-TEMPORALE",
    "BRN-SUPERIOR-TEMPORAL-GYRUS",
    "BRN-TEMPORAL-POLE",
    "BRN-TRANSVERSE-TEMPORAL-GYRUS",
]

ontology = set_parent(
    ontology,
    temporal_ids,
    "BRN-TEM"
)

# ------------------------------------------------------------
# 6. Occipital lobe
# ------------------------------------------------------------

occipital_ids = [
    "BRN-CALCARINE-CORTEX",
    "BRN-INFERIOR-OCCIPITAL-GYRUS",
    "BRN-LINGUAL-GYRUS",
    "BRN-MIDDLE-OCCIPITAL-GYRUS",
    "BRN-OCCIPITAL-FUSIFORM-GYRUS",
    "BRN-OCCIPITAL-POLE",
    "BRN-SUPERIOR-OCCIPITAL-GYRUS",
]

ontology = set_parent(
    ontology,
    occipital_ids,
    "BRN-OCC"
)

# ------------------------------------------------------------
# 7. Limbic region
# ------------------------------------------------------------

limbic_ids = [
    "BRN-ANTERIOR-CINGULATE-GYRUS",
    "BRN-MIDDLE-CINGULATE-GYRUS",
    "BRN-POSTERIOR-CINGULATE-GYRUS",
    "BRN-PARAHIPPOCAMPAL-GYRUS",
    "BRN-SUBCALLOSAL-AREA",
]

ontology = set_parent(
    ontology,
    limbic_ids,
    "BRN-LIM"
)

# ------------------------------------------------------------
# 8. Insula
# ------------------------------------------------------------

insula_ids = [
    "BRN-ANTERIOR-INSULA",
    "BRN-POSTERIOR-INSULA",
]

ontology = set_parent(
    ontology,
    insula_ids,
    "BRN-INS"
)

ontology["ontology_level"] = ontology["parent_id"].map(
    {
        "BRN": 1,
        "BRN-CER": 2,
        "BRN-CER-COR": 3,
        "BRN-FRO": 4,
        "BRN-PAR": 4,
        "BRN-TEM": 4,
        "BRN-OCC": 4,
        "BRN-LIM": 4,
        "BRN-INS": 4,
    }
).fillna(5).astype(int)

canonical_concepts_v2 = ontology

print("Total concepts:", len(canonical_concepts_v2))


In [ ]:
# ============================================================
# NON-CORTICAL HIERARCHY
# ============================================================

# Ventricular system
ventricle_ids = [
    "BRN-3RD-VENTRICLE",
    "BRN-4TH-VENTRICLE",
    "BRN-INF-LAT-VENT",
    "BRN-LATERAL-VENTRICLE",
]

# Deep gray matter
deep_gray_ids = [
    "BRN-ACCUMBENS",
    "BRN-AMYGDALA",
    "BRN-CAUDATE",
    "BRN-PALLIDUM",
    "BRN-PUTAMEN",
    "BRN-THALAMUS-PROPER",
    "BRN-VENTRAL-DC",
]

# Cerebellum
cerebellum_ids = [
    "BRN-CEREBELLUM-EXTERIOR",
    "BRN-CEREBELLUM-WHITE-MATTER",
    "BRN-CEREBELLAR-VERMAL-LOBULES-I-V",
    "BRN-CEREBELLAR-VERMAL-LOBULES-VI-VII",
    "BRN-CEREBELLAR-VERMAL-LOBULES-VIII-X",
]

# Brainstem
brainstem_ids = [
    "BRN-BRAIN-STEM",
]

# White matter
white_matter_ids = [
    "BRN-CEREBRAL-WHITE-MATTER",
]

# Apply parents
canonical_concepts_v2 = set_parent(
    canonical_concepts_v2, ventricle_ids, "BRN-VEN"
)

canonical_concepts_v2 = set_parent(
    canonical_concepts_v2, deep_gray_ids, "BRN-DGM"
)

canonical_concepts_v2 = set_parent(
    canonical_concepts_v2, cerebellum_ids, "BRN-CBL"
)

canonical_concepts_v2 = set_parent(
    canonical_concepts_v2, brainstem_ids, "BRN-BST-GRP"
)

canonical_concepts_v2 = set_parent(
    canonical_concepts_v2, white_matter_ids, "BRN-WM"
)


In [ ]:
# ============================================================
# FINISH REMAINING V2 HIERARCHY
# ============================================================

# Deep gray matter
deep_gray = [
    "BRN-ACC",
    "BRN-AMY",
    "BRN-CAU",
    "BRN-PAL",
    "BRN-PUT",
    "BRN-THA",
    "BRN-VDC",
]

# Limbic structures
limbic = [
    "BRN-HIP",
    "BRN-ENTORHINAL-AREA",
]

# Basal forebrain
basal_forebrain = [
    "BRN-BASAL-FOREBRAIN",
]

# Cerebellum
cerebellum = [
    "BRN-CB-EXT",
    "BRN-CB-WM",
]

# Brainstem
brainstem = [
    "BRN-BST",
]

# Cerebral white matter
white_matter = [
    "BRN-CER-WM",
]

# Ventricular system
ventricles = [
    "BRN-INF-LAT-V",
    "BRN-LAT-V",
]

# Cerebral cortex
# These are cortical regions that weren't assigned a lobe yet.
cortical_remaining = [
    "BRN-CUNEUS",
    "BRN-SUPERIOR-PARIETAL-LOBULE",
    "BRN-OPIFG-OPERCULAR-PART-OF-THE-IFG",
    "BRN-ORIFG-ORBITAL-PART-OF-THE-IFG",
    "BRN-TRIFG-TRIANGULAR-PART-OF-THE-IFG",
]

# Apply
canonical_concepts_v2 = set_parent(
    canonical_concepts_v2, deep_gray, "BRN-DGM"
)

canonical_concepts_v2 = set_parent(
    canonical_concepts_v2, limbic, "BRN-LIM"
)

canonical_concepts_v2 = set_parent(
    canonical_concepts_v2, basal_forebrain, "BRN-DGM"
)

canonical_concepts_v2 = set_parent(
    canonical_concepts_v2, cerebellum, "BRN-CBL"
)

canonical_concepts_v2 = set_parent(
    canonical_concepts_v2, brainstem, "BRN-BST-GRP"
)

canonical_concepts_v2 = set_parent(
    canonical_concepts_v2, white_matter, "BRN-WM"
)

canonical_concepts_v2 = set_parent(
    canonical_concepts_v2, ventricles, "BRN-VEN"
)

canonical_concepts_v2 = set_parent(
    canonical_concepts_v2, cortical_remaining, "BRN-CER-COR"
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

canonical_concepts_v2.to_csv(
    "path/to/canonical_concepts_v2.csv",
    index=False
)

print("Saved V2")
print("Total concepts:", len(canonical_concepts_v2))

In [ ]:
# ============================================================
# FINALIZE CANONICAL BRAIN ONTOLOGY V2
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# Load current versions
# ------------------------------------------------------------

ontology = canonical_concepts_v2.copy()
mapping = source_mappings_full.copy()

# ------------------------------------------------------------
# Clean / standardize
# ------------------------------------------------------------

ontology = ontology.drop_duplicates(
    subset=["canonical_id"]
).reset_index(drop=True)

mapping = mapping.drop_duplicates(
    subset=["source", "source_version", "source_id"]
).reset_index(drop=True)

# Ensure required columns exist
required_ontology_cols = [
    "canonical_id",
    "preferred_name",
    "parent_id",
    "category",
    "ontology_level",
    "laterality_allowed",
    "description",
]

for col in required_ontology_cols:
    if col not in ontology.columns:
        ontology[col] = ""

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

ontology_ids = set(ontology["canonical_id"])

# Missing mappings
missing_mapping = mapping[
    mapping["canonical_id"].isna()
    | (mapping["canonical_id"] == "")
]

# Invalid mappings
invalid_mapping = mapping[
    ~mapping["canonical_id"].isin(ontology_ids)
]

# Invalid parents
invalid_parents = ontology[
    ontology["parent_id"].notna()
    & (ontology["parent_id"] != "")
    & ~ontology["parent_id"].isin(ontology_ids)
]

# Duplicate canonical IDs
duplicate_concepts = ontology[
    ontology["canonical_id"].duplicated(keep=False)
]

# ------------------------------------------------------------
# Print final report
# ------------------------------------------------------------

print("=" * 60)
print("CANONICAL BRAIN ONTOLOGY V2 — FINAL REPORT")
print("=" * 60)

print(f"Canonical concepts       : {len(ontology)}")
print(f"UNesT source mappings   : {len(mapping)}")
print(f"Missing mappings         : {len(missing_mapping)}")
print(f"Invalid mappings         : {len(invalid_mapping)}")
print(f"Invalid parent IDs       : {len(invalid_parents)}")
print(f"Duplicate concepts       : {len(duplicate_concepts)}")

print("\nMapping review status:")
print(mapping["review_status"].value_counts(dropna=False))

# ------------------------------------------------------------
# Save final files
# ------------------------------------------------------------

ontology_path = Path(
    "path/to/canonical_brain_ontology_v2_FINAL.csv"
)

mapping_path = Path(
    "path/to/source_mapping_MONAI_UNEST_v0.2.7_FINAL.csv"
)

ontology.to_csv(ontology_path, index=False)
mapping.to_csv(mapping_path, index=False)

print("\n" + "=" * 60)
print("FINAL FILES")
print("=" * 60)

print(ontology_path)
print(mapping_path)

print("\nOntology successfully finalized.")

In [ ]:
# ============================================================
# STEP 2 — GENERATE 3D MESHES FOR ALL UNesT LABELS
# ============================================================

from skimage.measure import marching_cubes

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

segmentation_path = (
    "path/to/MNI305/"
    "IXI351_UNEST_segmentation_MNI305.nii.gz"
)

output_dir = "path/to/MNI305/meshes"
os.makedirs(output_dir, exist_ok=True)

# ------------------------------------------------------------
# Load segmentation
# ------------------------------------------------------------

seg_img = nib.load(segmentation_path)
seg = seg_img.get_fdata().astype(np.uint8)

spacing = seg_img.header.get_zooms()[:3]

print("Segmentation:", seg.shape)
print("Spacing:", spacing)

# ------------------------------------------------------------
# Use your finalized ontology + UNesT mapping
# ------------------------------------------------------------

ontology = canonical_concepts_v2.copy()
mapping = source_mappings_full.copy()

# ------------------------------------------------------------
# Generate meshes
# ------------------------------------------------------------

mesh_records = []

for label_id in range(1, 133):

    # Find UNesT mapping
    row = mapping[
        mapping["source_id"] == label_id
    ]

    if len(row) == 0:
        print(f"Skipping {label_id}: no mapping")
        continue

    row = row.iloc[0]

    source_name = row["source_name"]
    canonical_id = row["canonical_id"]
    laterality = row["laterality"]

    # --------------------------------------------------------
    # Mask
    # --------------------------------------------------------

    mask = seg == label_id

    voxel_count = int(mask.sum())

    if voxel_count == 0:
        print(f"Skipping {label_id}: no voxels")
        continue

    # --------------------------------------------------------
    # Marching cubes
    # --------------------------------------------------------

    try:
        vertices, faces, normals, values = marching_cubes(
            mask.astype(np.uint8),
            level=0.5,
            spacing=spacing
        )
    except Exception as e:
        print(f"Failed label {label_id}: {e}")
        continue

    # --------------------------------------------------------
    # Convert voxel coordinates → MNI305 world coordinates
    # --------------------------------------------------------

    vertices_world = nib.affines.apply_affine(
        seg_img.affine,
        vertices
    )

    # --------------------------------------------------------
    # Create mesh
    # --------------------------------------------------------

    mesh = trimesh.Trimesh(
        vertices=vertices_world,
        faces=faces,
        process=True
    )

    # Cleanup
    _, unique_indices = trimesh.grouping.unique_rows(mesh.faces)
    mesh.update_faces(unique_indices)

    # Fix orientation
    if mesh.volume < 0:
        mesh.invert()

    # --------------------------------------------------------
    # Filename
    # --------------------------------------------------------

    safe_name = (
        source_name
        .replace("/", "-")
        .replace(" ", "_")
        .replace(":", "-")
    )

    filename = (
        f"{label_id:03d}_"
        f"{canonical_id}_"
        f"{laterality}_"
        f"{safe_name}.glb"
    )

    filepath = os.path.join(
        output_dir,
        filename
    )

    # --------------------------------------------------------
    # Export
    # --------------------------------------------------------

    mesh.export(filepath)

    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    mesh_records.append({
        "source_id": int(label_id),
        "source_name": source_name,
        "canonical_id": canonical_id,
        "laterality": laterality,
        "voxel_count": voxel_count,
        "vertices": len(mesh.vertices),
        "faces": len(mesh.faces),
        "watertight": bool(mesh.is_watertight),
        "volume_mm3": float(mesh.volume),
        "mesh_file": filepath
    })

    print(
        f"{label_id:3d} | "
        f"{canonical_id:35s} | "
        f"{laterality:5s} | "
        f"voxels={voxel_count:7d} | "
        f"faces={len(mesh.faces):7d}"
    )

# ------------------------------------------------------------
# Save mesh registry
# ------------------------------------------------------------

mesh_registry = pd.DataFrame(mesh_records)

registry_path = (
    "path/to/MNI305/mesh_registry.csv"
)

mesh_registry.to_csv(
    registry_path,
    index=False
)

print("\n" + "=" * 60)
print("3D MESH GENERATION COMPLETE")
print("=" * 60)

print("Meshes generated:", len(mesh_registry))
print("Output directory:", output_dir)
print("Registry:", registry_path)

In [ ]:
# ============================================================
# STEP 3 — MESH QUALITY CONTROL
# ============================================================


registry_path = "path/to/MNI305/mesh_registry.csv"

df = pd.read_csv(registry_path)

# ------------------------------------------------------------
# Check every mesh
# ------------------------------------------------------------

quality = []

for _, row in df.iterrows():

    path = row["mesh_file"]

    mesh = trimesh.load_mesh(path, process=False)

    quality.append({
        "source_id": row["source_id"],
        "canonical_id": row["canonical_id"],
        "laterality": row["laterality"],
        "vertices": len(mesh.vertices),
        "faces": len(mesh.faces),
        "watertight": mesh.is_watertight,
        "winding_consistent": mesh.is_winding_consistent,
        "components": len(mesh.split(only_watertight=False)),
        "volume_mm3": abs(float(mesh.volume)),
        "file_size_MB": os.path.getsize(path) / (1024**2)
    })

quality_df = pd.DataFrame(quality)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("=" * 60)
print("MESH QUALITY REPORT")
print("=" * 60)

print("Meshes:", len(quality_df))
print("Watertight:",
      quality_df["watertight"].sum(),
      "/",
      len(quality_df))

print("Winding consistent:",
      quality_df["winding_consistent"].sum(),
      "/",
      len(quality_df))

print(
    "Total vertices:",
    f"{quality_df['vertices'].sum():,}"
)

print(
    "Total faces:",
    f"{quality_df['faces'].sum():,}"
)

print(
    "Total size:",
    f"{quality_df['file_size_MB'].sum():.2f} MB"
)

# ------------------------------------------------------------
# Problematic meshes
# ------------------------------------------------------------

problems = quality_df[
    (~quality_df["watertight"])
    | (~quality_df["winding_consistent"])
    | (quality_df["components"] > 1)
]

print("\nPotentially problematic meshes:", len(problems))

if len(problems):
    display(problems)

# ------------------------------------------------------------
# Largest meshes
# ------------------------------------------------------------

print("\nLargest meshes:")

display(
    quality_df[
        [
            "source_id",
            "canonical_id",
            "vertices",
            "faces",
            "file_size_MB"
        ]
    ]
    .sort_values("faces", ascending=False)
    .head(15)
)

# ------------------------------------------------------------
# Save QC report
# ------------------------------------------------------------

qc_path = "path/to/MNI305/mesh_quality_report.csv"

quality_df.to_csv(qc_path, index=False)

print("\nQC report saved:", qc_path)

In [ ]:
# ============================================================
# STEP 4 — OPTIMIZE 3D MESHES
# ============================================================
registry_path = "path/to/MNI305/mesh_registry.csv"
df = pd.read_csv(registry_path)

input_dir = "path/to/MNI305/meshes"
output_dir = "path/to/MNI305/meshes_optimized"

os.makedirs(output_dir, exist_ok=True)

optimized_records = []

# ------------------------------------------------------------
# Target face count
# ------------------------------------------------------------

TARGET_FACES = 10000

# ------------------------------------------------------------
# Process meshes
# ------------------------------------------------------------

for _, row in df.iterrows():

    source_id = int(row["source_id"])
    canonical_id = row["canonical_id"]
    laterality = row["laterality"]

    input_path = row["mesh_file"]

    mesh = trimesh.load_mesh(
        input_path,
        process=True
    )

    # --------------------------------------------------------
    # Fix orientation
    # --------------------------------------------------------

    if mesh.volume < 0:
        mesh.invert()

    # --------------------------------------------------------
    # Remove very small disconnected components
    # --------------------------------------------------------

    components = mesh.split(
        only_watertight=False
    )

    if len(components) > 1:

        # Keep components with meaningful surface area
        components = sorted(
            components,
            key=lambda x: x.area,
            reverse=True
        )

        main_area = components[0].area

        components = [
            c for c in components
            if c.area >= main_area * 0.01
        ]

        mesh = trimesh.util.concatenate(
            components
        )

    # --------------------------------------------------------
    # Decimate large meshes
    # --------------------------------------------------------

    original_faces = len(mesh.faces)

    if original_faces > TARGET_FACES:

        try:
            mesh = mesh.simplify_quadric_decimation(
                face_count=TARGET_FACES
            )
        except Exception as e:
            print(
                f"Decimation failed for "
                f"{source_id}: {e}"
            )

    # --------------------------------------------------------
    # Final cleanup
    # --------------------------------------------------------

    _, unique_indices = trimesh.grouping.unique_rows(mesh.faces)
    mesh.update_faces(unique_indices)


    if mesh.volume < 0:
        mesh.invert()

    # --------------------------------------------------------
    # Output
    # --------------------------------------------------------

    filename = os.path.basename(input_path)

    output_path = os.path.join(
        output_dir,
        filename
    )

    mesh.export(output_path)

    optimized_records.append({
        "source_id": source_id,
        "canonical_id": canonical_id,
        "laterality": laterality,
        "original_faces": original_faces,
        "optimized_faces": len(mesh.faces),
        "original_vertices": int(row["vertices"]),
        "optimized_vertices": len(mesh.vertices),
        "watertight": mesh.is_watertight,
        "volume_mm3": abs(float(mesh.volume)),
        "file_size_MB":
            os.path.getsize(output_path) / (1024**2),
        "mesh_file": output_path
    })

    print(
        f"{source_id:3d} | "
        f"{canonical_id:35s} | "
        f"{original_faces:7,d} → "
        f"{len(mesh.faces):7,d}"
    )

# ------------------------------------------------------------
# Save optimized registry
# ------------------------------------------------------------

optimized_df = pd.DataFrame(
    optimized_records
)

optimized_registry = (
    "path/to/MNI305/"
    "mesh_registry_optimized.csv"
)

optimized_df.to_csv(
    optimized_registry,
    index=False
)

print("\n" + "=" * 60)
print("OPTIMIZATION COMPLETE")
print("=" * 60)

print("Meshes:", len(optimized_df))
print(
    "Original faces:",
    f"{optimized_df['original_faces'].sum():,}"
)
print(
    "Optimized faces:",
    f"{optimized_df['optimized_faces'].sum():,}"
)
print(
    "Total GLB size:",
    f"{optimized_df['file_size_MB'].sum():.2f} MB"
)

print("\nSaved:", optimized_registry)

In [ ]:
# ============================================================
# STEP 6 — REMOVE FLOATING MESH COMPONENTS
# ============================================================

input_dir = "path/to/MNI305/meshes_optimized"
output_dir = "path/to/MNI305/meshes_clean"

os.makedirs(output_dir, exist_ok=True)

registry_path = (
    "path/to/MNI305/mesh_registry_optimized.csv"
)

df = pd.read_csv(registry_path)

clean_records = []

for _, row in df.iterrows():

    input_path = row["mesh_file"]

    if not os.path.exists(input_path):
        continue

    mesh = trimesh.load_mesh(
        input_path,
        process=True
    )

    # --------------------------------------------------------
    # Find connected components
    # --------------------------------------------------------

    components = mesh.split(
        only_watertight=False
    )

    if len(components) > 1:

        # Sort by number of faces
        components = sorted(
            components,
            key=lambda x: len(x.faces),
            reverse=True
        )

        # Keep main component + meaningful components
        main_faces = len(components[0].faces)

        components = [
            c for c in components
            if len(c.faces) >= main_faces * 0.01
        ]

        mesh = trimesh.util.concatenate(
            components
        )

    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------

    _, unique_indices = trimesh.grouping.unique_rows(mesh.faces)
    mesh.update_faces(unique_indices)

    if mesh.volume < 0:
        mesh.invert()

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    filename = os.path.basename(input_path)

    output_path = os.path.join(
        output_dir,
        filename
    )

    mesh.export(output_path)

    clean_records.append({
        "source_id": int(row["source_id"]),
        "canonical_id": row["canonical_id"],
        "laterality": row["laterality"],
        "vertices": len(mesh.vertices),
        "faces": len(mesh.faces),
        "watertight": mesh.is_watertight,
        "volume_mm3": abs(float(mesh.volume)),
        "mesh_file": output_path
    })

clean_df = pd.DataFrame(clean_records)

clean_registry = (
    "path/to/MNI305/"
    "mesh_registry_clean.csv"
)

clean_df.to_csv(
    clean_registry,
    index=False
)

print("=" * 60)
print("MESH CLEANING COMPLETE")
print("=" * 60)

print("Meshes:", len(clean_df))
print("Registry:", clean_registry)

In [ ]:
# ============================================================
# STEP 7 — BUILD CLEAN COMPLETE BRAIN
# ============================================================


registry_path = (
    "path/to/MNI305/"
    "mesh_registry_clean.csv"
)

output_path = (
    "path/to/MNI305/"
    "IXI351_UNEST_CanonicalBrain_CLEAN.glb"
)

df = pd.read_csv(registry_path)

scene = trimesh.Scene()

for _, row in df.iterrows():

    path = row["mesh_file"]

    if not os.path.exists(path):
        continue

    mesh = trimesh.load_mesh(
        path,
        process=False
    )

    name = (
        f"{row['canonical_id']}_"
        f"{row['laterality']}_"
        f"{int(row['source_id']):03d}"
    )

    mesh.metadata = {
        "source_id": int(row["source_id"]),
        "canonical_id": row["canonical_id"],
        "laterality": row["laterality"]
    }

    scene.add_geometry(
        mesh,
        node_name=name,
        geom_name=name
    )

scene.export(
    output_path,
    file_type="glb"
)

print("Objects:", len(scene.geometry))
print("Saved:", output_path)
print(
    "Size:",
    f"{os.path.getsize(output_path)/(1024**2):.2f} MB"
)

In [ ]:
# ============================================================
# STEP 8 — CREATE VIEWER METADATA MANIFEST
# ============================================================

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

ontology_path = (
    "path/to/canonical_concepts_v2.csv"
)

mapping_path = (
    "path/to/"
    "source_mapping_MONAI_UNEST_v0.2.7_FINAL.csv"
)

registry_path = (
    "path/to/MNI305/"
    "mesh_registry_clean.csv"
)

manifest_path = (
    "path/to/MNI305/"
    "brain_manifest.json"
)

# ------------------------------------------------------------
# Load
# ------------------------------------------------------------

ontology = pd.read_csv(ontology_path)
mapping = pd.read_csv(mapping_path)
registry = pd.read_csv(registry_path)

# ------------------------------------------------------------
# Convert NaN → None
# ------------------------------------------------------------

def clean_value(value):

    if pd.isna(value):
        return None

    if hasattr(value, "item"):
        return value.item()

    return value


# ------------------------------------------------------------
# Build ontology lookup
# ------------------------------------------------------------

ontology_lookup = {}

for _, row in ontology.iterrows():

    ontology_lookup[
        row["canonical_id"]
    ] = {
        "canonical_id":
            clean_value(row["canonical_id"]),

        "preferred_name":
            clean_value(row["preferred_name"]),

        "parent_id":
            clean_value(row["parent_id"]),

        "category":
            clean_value(row["category"]),

        "ontology_level":
            clean_value(row["ontology_level"]),

        "laterality_allowed":
            clean_value(row["laterality_allowed"]),

        "description":
            clean_value(row["description"])
    }


# ------------------------------------------------------------
# Build structure records
# ------------------------------------------------------------

structures = []

for _, row in registry.iterrows():

    source_id = int(row["source_id"])
    canonical_id = row["canonical_id"]

    ontology_info = ontology_lookup.get(
        canonical_id,
        {}
    )

    structure = {

        # Viewer object identification
        "object_name": (
            f"{canonical_id}_"
            f"{row['laterality']}_"
            f"{source_id:03d}"
        ),

        # Source information
        "source": "MONAI_UNesT",
        "source_version": "0.2.7",
        "source_id": source_id,
        "source_name": clean_value(
            mapping[
                mapping["source_id"] == source_id
            ]["source_name"].iloc[0]
        ),

        # Canonical ontology
        "canonical_id": canonical_id,

        "preferred_name":
            ontology_info.get(
                "preferred_name"
            ),

        "parent_id":
            ontology_info.get(
                "parent_id"
            ),

        "category":
            ontology_info.get(
                "category"
            ),

        "ontology_level":
            ontology_info.get(
                "ontology_level"
            ),

        # Laterality
        "laterality":
            clean_value(row["laterality"]),

        # Mesh
        "mesh_file":
            os.path.basename(
                row["mesh_file"]
            ),

        "vertices":
            int(row["vertices"]),

        "faces":
            int(row["faces"]),

        "volume_mm3":
            float(row["volume_mm3"]),

        "watertight":
            bool(row["watertight"])
    }

    structures.append(structure)


# ------------------------------------------------------------
# Final manifest
# ------------------------------------------------------------

manifest = {

    "schema_version": "1.0",

    "model": {
        "name": "MONAI Whole Brain Large UNesT",
        "version": "0.2.7",
        "source": "MONAI",
        "label_count": 133
    },

    "space": {
        "name": "MNI305",
        "coordinate_system": "RAS",
        "voxel_spacing_mm": [1.0, 1.0, 1.0]
    },

    "ontology": {
        "name": "Canonical Brain Ontology",
        "version": "2.0"
    },

    "brain_model": {
        "glb_file":
            "IXI351_UNEST_CanonicalBrain_CLEAN.glb",

        "structure_count":
            len(structures)
    },

    "structures": structures
}


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


print("=" * 60)
print("BRAIN METADATA MANIFEST CREATED")
print("=" * 60)

print("Structures:", len(structures))
print("Manifest:", manifest_path)

print(
    "Size:",
    f"{os.path.getsize(manifest_path) / 1024:.2f} KB"
)

In [ ]:
# ============================================================
# STEP 9 — FINALIZE HIERARCHY LEVELS + VERIFY GLB/METADATA
# ============================================================

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

ontology_path = (
    "path/to/canonical_concepts_v2.csv"
)

manifest_path = (
    "path/to/MNI305/brain_manifest.json"
)

glb_path = (
    "path/to/MNI305/"
    "IXI351_UNEST_CanonicalBrain_CLEAN.glb"
)

# ------------------------------------------------------------
# Load
# ------------------------------------------------------------

ontology = pd.read_csv(ontology_path)

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

# ------------------------------------------------------------
# Calculate hierarchy levels recursively
# ------------------------------------------------------------

parent_map = dict(
    zip(
        ontology["canonical_id"],
        ontology["parent_id"]
    )
)

level_cache = {}

def get_level(canonical_id, visited=None):

    if visited is None:
        visited = set()

    if canonical_id in level_cache:
        return level_cache[canonical_id]

    if canonical_id == "BRN":
        level_cache[canonical_id] = 0
        return 0

    if canonical_id in visited:
        raise ValueError(
            f"Hierarchy cycle detected at {canonical_id}"
        )

    visited.add(canonical_id)

    parent = parent_map.get(canonical_id)

    if pd.isna(parent) or parent == "":
        level = 0
    else:
        level = get_level(parent, visited) + 1

    level_cache[canonical_id] = level

    return level


ontology["ontology_level"] = ontology[
    "canonical_id"
].apply(get_level)

# ------------------------------------------------------------
# Update manifest hierarchy levels
# ------------------------------------------------------------

ontology_lookup = ontology.set_index(
    "canonical_id"
).to_dict("index")

for structure in manifest["structures"]:

    cid = structure["canonical_id"]

    if cid in ontology_lookup:

        structure["ontology_level"] = int(
            ontology_lookup[cid]["ontology_level"]
        )

# ------------------------------------------------------------
# Save corrected ontology
# ------------------------------------------------------------

ontology.to_csv(
    ontology_path,
    index=False
)

# Save corrected manifest

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# Verify GLB objects
# ------------------------------------------------------------

scene = trimesh.load(
    glb_path,
    force="scene"
)

glb_objects = set(scene.geometry.keys())

manifest_objects = set(
    s["object_name"]
    for s in manifest["structures"]
)

missing_from_glb = manifest_objects - glb_objects
missing_from_manifest = glb_objects - manifest_objects

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 60)
print("VIEWER DATA VALIDATION")
print("=" * 60)

print("Ontology concepts:", len(ontology))
print("Manifest structures:", len(manifest["structures"]))
print("GLB objects:", len(glb_objects))

print("\nManifest objects missing from GLB:",
      len(missing_from_glb))

print("GLB objects missing from manifest:",
      len(missing_from_manifest))

assert len(missing_from_glb) == 0
assert len(missing_from_manifest) == 0

print("\n✓ GLB ↔ metadata mapping is valid")
print("✓ Hierarchy levels recalculated")

In [ ]:
# ============================================================
# STEP 10 — CREATE VIEWER DATA PACKAGE
# ============================================================

import os
import shutil

viewer_dir = "path/to/brain_viewer_data"

os.makedirs(viewer_dir, exist_ok=True)

# Copy complete brain
shutil.copy2(
    "path/to/MNI305/"
    "IXI351_UNEST_CanonicalBrain_CLEAN.glb",

    viewer_dir
    + "/brain.glb"
)

# Copy manifest
shutil.copy2(
    "path/to/MNI305/"
    "brain_manifest.json",

    viewer_dir
    + "/manifest.json"
)

# Copy ontology
shutil.copy2(
    "path/to/"
    "canonical_concepts_v2.csv",

    viewer_dir
    + "/ontology.csv"
)

# Copy source mapping
shutil.copy2(
    "path/to/"
    "source_mapping_MONAI_UNEST_v0.2.7_FINAL.csv",

    viewer_dir
    + "/source_mapping.csv"
)

print("=" * 60)
print("VIEWER DATA PACKAGE")
print("=" * 60)

for file in sorted(os.listdir(viewer_dir)):

    path = os.path.join(viewer_dir, file)

    print(
        f"{file:30s}"
        f"{os.path.getsize(path)/(1024**2):8.2f} MB"
    )

print("\nLocation:")
print(viewer_dir)

In [ ]:
# ============================================================
# IXI BATCH PIPELINE
# ============================================================
import warnings
warnings.filterwarnings('ignore')
import sys
import os, json, shutil, gc
import numpy as np
import pandas as pd
import nibabel as nib
import ants
import torch
import trimesh

from glob import glob
from scipy import ndimage
from skimage.measure import marching_cubes

bundle_root = "path/to/wholeBrainSeg_Large_UNEST_segmentation/wholeBrainSeg_Large_UNEST_segmentation"

if bundle_root not in sys.path:
    sys.path.insert(0, bundle_root)
    
from scripts.networks.unest_base_patch_4 import UNesT
from monai.bundle import ConfigParser

BUNDLE_DIR = "path/to/wholeBrainSeg_Large_UNEST_segmentation/wholeBrainSeg_Large_UNEST_segmentation"

# Load model architecture from bundle
parser = ConfigParser()
parser.read_config(
    f"{BUNDLE_DIR}/configs/inference.json"
)

model = parser.get_parsed_content("network")
model = model.to("cuda" if torch.cuda.is_available() else "cpu")

# Load trained weights
checkpoint = torch.load(
    f"{BUNDLE_DIR}/models/model.pt",
    map_location="cuda" if torch.cuda.is_available() else "cpu"
)

model.load_state_dict(checkpoint["model"])
model.eval()

print("✓ UNesT model loaded")
print("Device:", next(model.parameters()).device)



from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd,
    NormalizeIntensityd, EnsureTyped
)
from monai.inferers import SlidingWindowInferer
# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

IXI_DIR = "path/to/datasets/jackontreesandminds/ixi-t1-basemri"
MNI305_TEMPLATE = "path/to/datasets/jackontreesandminds/mni305-template/average305_t1_tal_lin.nii"
ONTOLOGY_CSV = "path/to/datasets/jackontreesandminds/required-files/canonical_brain_ontology_v2_FINAL.csv"
MAPPING_CSV = "path/to/datasets/jackontreesandminds/required-files/source_mapping_MONAI_UNEST_v0.2.7_FINAL.csv"

BATCH_OUTPUT = "path/to/IXI_UNEST_BATCH"

os.makedirs(BATCH_OUTPUT, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)
model.eval()

print("Device:", device)


# ------------------------------------------------------------
# LOAD ONTOLOGY + MAPPING
# ------------------------------------------------------------

ontology = pd.read_csv(ONTOLOGY_CSV)
mapping = pd.read_csv(MAPPING_CSV)

mapping["source_id"] = mapping["source_id"].astype(int)

mapping_lookup = mapping.set_index("source_id").to_dict("index")
ontology_lookup = ontology.set_index("canonical_id").to_dict("index")

print("Ontology concepts:", len(ontology))
print("Source mappings:", len(mapping))


# ------------------------------------------------------------
# MONAI PREPROCESSOR + INFERER
# ------------------------------------------------------------

preprocess = Compose([
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    NormalizeIntensityd(
        keys=["image"],
        nonzero=True,
        channel_wise=True
    ),
    EnsureTyped(keys=["image"])
])

inferer = SlidingWindowInferer(
    roi_size=(96, 96, 96),
    sw_batch_size=1,
    overlap=0.7,
    mode="constant"
)


# ============================================================
# SINGLE SUBJECT
# ============================================================

def process_subject(input_path, subject_id):

    print("\n" + "=" * 60)
    print("SUBJECT:", subject_id)
    print("=" * 60)

    subject_dir = os.path.join(
        BATCH_OUTPUT,
        subject_id
    )

    ras_dir = os.path.join(subject_dir, "RAS")
    mni_dir = os.path.join(subject_dir, "MNI305")
    transform_dir = os.path.join(mni_dir, "transforms")
    mesh_dir = os.path.join(mni_dir, "meshes")

    for d in [ras_dir, mni_dir, transform_dir, mesh_dir]:
        os.makedirs(d, exist_ok=True)


    # --------------------------------------------------------
    # 1. RAS
    # --------------------------------------------------------

    ras_path = os.path.join(
        ras_dir,
        f"{subject_id}_T1_RAS.nii.gz"
    )

    img = nib.load(input_path)
    ras_img = nib.as_closest_canonical(img)
    nib.save(ras_img, ras_path)

    print("1. RAS:", ras_img.shape)


    # --------------------------------------------------------
    # 2. MNI305 REGISTRATION
    # --------------------------------------------------------

    fixed = ants.image_read(MNI305_TEMPLATE)
    moving = ants.image_read(ras_path)

    registration = ants.registration(
        fixed=fixed,
        moving=moving,
        type_of_transform="Affine"
    )

    registered_path = os.path.join(
        mni_dir,
        f"{subject_id}_T1_MNI305.nii.gz"
    )

    ants.image_write(
        registration["warpedmovout"],
        registered_path
    )

    transform_path = os.path.join(
        transform_dir,
        f"{subject_id}_to_MNI305_Affine.mat"
    )

    shutil.copy2(
        registration["fwdtransforms"][0],
        transform_path
    )

    print("2. Registered to MNI305")


    # --------------------------------------------------------
    # 3. UNEST
    # --------------------------------------------------------

    data = preprocess({
        "image": registered_path
    })

    image = data["image"].unsqueeze(0).to(device)

    with torch.inference_mode():
        logits = inferer(image, model)

    labels = (
        logits.argmax(dim=1)[0]
        .cpu()
        .numpy()
        .astype(np.uint8)
    )

    print(
        "3. UNesT:",
        len(np.unique(labels)),
        "labels"
    )


    # --------------------------------------------------------
    # 4. SAVE SEGMENTATION
    # --------------------------------------------------------

    reference = nib.load(registered_path)

    segmentation_path = os.path.join(
        mni_dir,
        f"{subject_id}_UNEST_segmentation_MNI305.nii.gz"
    )

    seg_img = nib.Nifti1Image(
        labels,
        reference.affine,
        reference.header.copy()
    )

    seg_img.set_data_dtype(np.uint8)

    nib.save(
        seg_img,
        segmentation_path
    )


    # --------------------------------------------------------
    # 5. MERGE SOURCE LABELS
    #    canonical_id + laterality
    # --------------------------------------------------------

    groups = {}

    for source_id, info in mapping_lookup.items():

        if source_id == 0:
            continue

        canonical_id = info["canonical_id"]
        laterality = info["laterality"]

        key = (
            canonical_id,
            laterality
        )

        if key not in groups:
            groups[key] = []

        groups[key].append(source_id)


    # --------------------------------------------------------
    # 6. CREATE CANONICAL MESHES
    # --------------------------------------------------------
    affine = reference.affine

    registry = []

    for (canonical_id, laterality), source_ids in groups.items():

        # Merge all source labels belonging to this concept
        mask = np.isin(
            labels,
            source_ids
        )



        # Largest connected component
        cc, n = ndimage.label(
            mask,
            structure=np.ones((3, 3, 3))
        )

        if n > 1:
            counts = np.bincount(cc.ravel())
            counts[0] = 0
            mask = cc == counts.argmax()


        voxel_count = int(mask.sum())

        if voxel_count < 20:
            continue
        volume_mm3 = float(voxel_count)
        # Marching cubes
        verts, faces, _, _ = marching_cubes(
            mask.astype(np.uint8),
            level=0.5
        )


        # voxel → MNI world coordinates
        verts_world = nib.affines.apply_affine(
            affine,
            verts
        )


        mesh = trimesh.Trimesh(
            vertices=verts_world,
            faces=faces,
            process=True
        )

        _, unique_indices = trimesh.grouping.unique_rows(mesh.faces)
        mesh.update_faces(unique_indices)

        if mesh.is_watertight and mesh.volume < 0:
            mesh.invert()


        concept = ontology_lookup.get(
            canonical_id,
            {}
        )

        preferred_name = concept.get(
            "preferred_name",
            canonical_id
        )

        object_name = (
            f"{canonical_id}_{laterality}"
        )

        mesh_file = (
            object_name + ".glb"
        )

        mesh_path = os.path.join(
            mesh_dir,
            mesh_file
        )

        mesh.export(mesh_path)


        registry.append({

            "object_name": object_name,

            "source": "MONAI_UNesT",

            "source_version": "0.2.7",

            "source_ids": ",".join(
                map(str, source_ids)
            ),

            "canonical_id": canonical_id,

            "preferred_name": preferred_name,

            "parent_id": concept.get(
                "parent_id"
            ),

            "category": concept.get(
                "category"
            ),

            "ontology_level": concept.get(
                "ontology_level"
            ),

            "laterality": laterality,

            "mesh_file": mesh_file,

            "voxel_count": voxel_count,

            "vertices": len(mesh.vertices),

            "faces": len(mesh.faces),

            "volume_mm3": volume_mm3,

            "watertight": bool(
                mesh.is_watertight
            )
        })


    registry = pd.DataFrame(registry)

    registry_path = os.path.join(
        mni_dir,
        "mesh_registry.csv"
    )

    registry.to_csv(
        registry_path,
        index=False
    )
    # --------------------------------------------------------
    # ANALYSIS-READY MEASUREMENTS
    # --------------------------------------------------------
    
    analysis_df = registry[
        [
            "canonical_id",
            "preferred_name",
            "laterality",
            "voxel_count",
            "volume_mm3",
            "parent_id",
            "category",
            "ontology_level"
        ]
    ].copy()
    
    analysis_df.insert(
        0,
        "model_version",
        "0.2.7"
    )
    
    analysis_df.insert(
        0,
        "model",
        "UNesT"
    )
    
    analysis_df.insert(
        0,
        "subject_id",
        subject_id
    )
    
    analysis_path = os.path.join(
        mni_dir,
        f"{subject_id}_structure_measurements.csv"
    )
    
    analysis_df.to_csv(
        analysis_path,
        index=False
    )
    
    print(
        "Analysis table:",
        analysis_path
    )

    # --------------------------------------------------------
    # 7. COMBINED GLB
    # --------------------------------------------------------
    scene = trimesh.Scene()

    for _, row in registry.iterrows():

        mesh_path = os.path.join(
            mesh_dir,
            row["mesh_file"]
        )

        mesh = trimesh.load(
            mesh_path,
            force="mesh"
        )

        scene.add_geometry(
            mesh,
            node_name=row["object_name"],
            geom_name=row["object_name"]
        )


    brain_glb = os.path.join(
        mni_dir,
        "brain.glb"
    )

    scene.export(brain_glb)


    # --------------------------------------------------------
    # 8. MANIFEST
    # --------------------------------------------------------

    manifest = {

        "schema": "1.0",

        "subject_id": subject_id,

        "model": {
            "name": "MONAI Whole Brain Large UNesT",
            "version": "0.2.7",
            "source": "MONAI",
            "label_count": 133
        },

        "space": {
            "name": "MNI305",
            "coordinate_system": "RAS",
            "voxel_spacing_mm": [1, 1, 1]
        },

        "ontology": {
            "name": "Canonical Brain Ontology",
            "version": "2.0"
        },

        "segmentation": os.path.basename(
            segmentation_path
        ),

        "registration": {
            "transform": os.path.relpath(
                transform_path,
                mni_dir
            )
        },

        "structures": registry.to_dict(
            orient="records"
        )
    }


    manifest_path = os.path.join(
        mni_dir,
        "manifest.json"
    )

    with open(manifest_path, "w") as f:
        json.dump(
            manifest,
            f,
            indent=2
        )


    # Free GPU memory
    del image, logits, data
    torch.cuda.empty_cache()
    gc.collect()


    print("4. Canonical structures:", len(registry))
    print("5. GLB:", brain_glb)
    print("✓ COMPLETE")

    return manifest


# ============================================================
#  BATCH LOOP
# ============================================================

raw_paths = (
    glob(os.path.join(IXI_DIR, "**", "*.nii"), recursive=True) +
    glob(os.path.join(IXI_DIR, "**", "*.nii.gz"), recursive=True)
)

# Filter to keep ONLY actual files (ignoring directories named .nii) and take the first 50
files = sorted([f for f in raw_paths if os.path.isfile(f)])[:50]
print("\nFound", len(files), "MRI files")


results = []

for i, input_path in enumerate(files):

    filename = os.path.basename(input_path)

    # IXI014-HH-1236-T1.nii.gz → IXI014
    subject_id = filename.split("-")[0]

    print(
        f"\n[{i+1}/{len(files)}] {subject_id}"
    )

    try:

        manifest = process_subject(
            input_path,
            subject_id
        )

        results.append({
            "subject_id": subject_id,
            "status": "SUCCESS",
            "structures": len(
                manifest["structures"]
            )
        })

    except Exception as e:

        print(
            "ERROR:",
            repr(e)
        )

        results.append({
            "subject_id": subject_id,
            "status": "FAILED",
            "structures": 0
        })

        torch.cuda.empty_cache()
        gc.collect()



# ============================================================
# FINAL SUMMARY
# ============================================================

results_df = pd.DataFrame(results)

summary_path = os.path.join(
    BATCH_OUTPUT,
    "batch_summary.csv"
)
measurement_files = glob(
    os.path.join(
        BATCH_OUTPUT,
        "*",
        "MNI305",
        "*_structure_measurements.csv"
    )
)

all_measurements = pd.concat(
    [pd.read_csv(f) for f in measurement_files],
    ignore_index=True
)

all_measurements.to_csv(
    os.path.join(
        BATCH_OUTPUT,
        "UNesT_structure_measurements_ALL.csv"
    ),
    index=False
)

print(
    "Total measurement rows:",
    len(all_measurements)
)

print(
    "Subjects:",
    all_measurements["subject_id"].nunique()
)
results_df.to_csv(
    summary_path,
    index=False
)

print("\n======================================")
print("BATCH COMPLETE")
print("======================================")
print(results_df["status"].value_counts())
print("Summary:", summary_path)